In [1]:
import torch
import torch.nn as nn

_ = torch.manual_seed(26)

In [2]:
# input & output initialization
x = torch.tensor([
    [1, 2, 3, 4, 5],
    [2, 3, 4, 5, 6],
    [3, 4, 5, 6, 7],
    [1, 3, 5, 7, 9],
    [0, 2, 4, 6, 8],
    [4, 5, 6, 7, 8],
    [5, 4, 3, 2, 1],
    [6, 5, 4, 3, 2],
    [9, 7, 5, 3, 1],
    [3, 1, 4, 1, 5],
    [2, 7, 1, 8, 2],
    [4, 4, 4, 4, 4],
], dtype=torch.float32)
# 1 if increasing, 0 if decreasing (mixed or flat also count as 0)
y = torch.tensor([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0], dtype=torch.float32).view(-1, 1)
input_size = x.shape[-1]
output_size = y.shape[-1]
hidden_size = 4

In [3]:
# model declaration
class MultiLayerPerceptron(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features=input_size, out_features=hidden_size),
            nn.ReLU(),
            nn.Linear(in_features=hidden_size, out_features=output_size),
        )

    def forward(self, input):
        return self.layers(input=input)

In [4]:
# model initialization
model = MultiLayerPerceptron(input_size=input_size, hidden_size=hidden_size, output_size=output_size)
for name, param in model.named_parameters():
    print(f"{name}: {param.data}")
# optimizer initialization
optimizer = torch.optim.AdamW(params=model.parameters(), lr=0.01)

layers.0.weight: tensor([[ 0.2964,  0.0505,  0.4152, -0.0012,  0.1648],
        [-0.2764, -0.4110, -0.0616,  0.3255,  0.2467],
        [-0.3507,  0.3441,  0.4413, -0.2675, -0.4247],
        [ 0.0901, -0.2548,  0.3812,  0.3647, -0.3334]])
layers.0.bias: tensor([-0.4340, -0.0949,  0.3931, -0.0442])
layers.2.weight: tensor([[-0.1120,  0.4025,  0.3006, -0.0022]])
layers.2.bias: tensor([-0.4586])


In [5]:
# training
epochs = 500
for epoch in range(epochs):
    # forward pass
    z = model(input=x)
    # calculate loss
    loss = nn.BCEWithLogitsLoss()(input=z, target=y)
    # backward pass
    optimizer.zero_grad()
    loss.backward()
    if epoch % 50 == 0:
        print(f"epoch {epoch} loss: {loss.item()}")
        for name, param in model.named_parameters():
            print(f"{name}.grad: {param.grad}")
    # update parameters
    optimizer.step()

epoch 0 loss: 0.6114358901977539
layers.0.weight.grad: tensor([[-2.1776e-02,  1.8525e-02,  8.0665e-02,  1.1726e-01,  1.7649e-01],
        [-2.2935e-01, -3.7036e-01, -5.1137e-01, -6.5237e-01, -7.9338e-01],
        [ 1.5809e-01,  1.2646e-01,  9.4828e-02,  6.3199e-02,  3.1571e-02],
        [-4.3046e-04,  3.6619e-04,  1.5946e-03,  2.3179e-03,  3.4889e-03]])
layers.0.bias.grad: tensor([ 0.0132, -0.1119,  0.0239,  0.0003])
layers.2.weight.grad: tensor([[-0.4570, -0.3733,  0.0300, -0.0662]])
layers.2.bias.grad: tensor([-0.1176])
epoch 50 loss: 0.1371208280324936
layers.0.weight.grad: tensor([[-0.0306, -0.0097,  0.0022,  0.0260,  0.0241],
        [ 0.0423,  0.0018, -0.0194, -0.0661, -0.0576],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0171,  0.0007, -0.0079, -0.0268, -0.0233]])
layers.0.bias.grad: tensor([-0.0073,  0.0115,  0.0000,  0.0046])
layers.2.weight.grad: tensor([[ 0.0846, -0.1082,  0.0000, -0.0847]])
layers.2.bias.grad: tensor([0.0197])
epoch 100 loss: 0.031003

In [6]:
# prediction
with torch.inference_mode():
    z = model(input=x)
# activation function (value > 0 is 1, else 0)
y_pred = (z > 0).float()
print(f"y_pred: {y_pred.view(-1)}")
print(f"y:      {y.view(-1)}")

y_pred: tensor([1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.])
y:      tensor([1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.])
